In [12]:
from sklearn.model_selection import train_test_split
import pandas as pd

train_data = pd.read_csv('./data/train.csv') # Importing training data

new_train_data = pd.read_csv("./data/train_new.csv")

train_data = pd.concat((train_data, new_train_data), axis = 1, join = "inner")

X_train = train_data.drop(["time", "Y1", "Y2"], axis = 1) # Losing the time and target columns

# Setting target variables
y1 = train_data["Y1"]

y2 = train_data["Y2"]

X_train, X_val, y1_train, y1_val, y2_train, y2_val = train_test_split(X_train, y1, y2)

In [13]:
from sklearn.decomposition import PCA

# Applying PCA to training set
X_train_scaled = (X_train.drop(["O", "P"], axis = 1) - X_train.drop(["O", "P"], axis = 1).mean(axis=0)) / X_train.drop(["O", "P"], axis = 1).std(axis=0)

pca = PCA(n_components= 3)

X_pca = pca.fit_transform(X_train_scaled)

X_train["PC1"] = X_pca[:,0]
X_train["PC2"] = X_pca[:,1]
X_train["PC3"] = X_pca[:,2]

In [14]:
loadings = pd.DataFrame(
    pca.components_.T,
    columns = ["PC1", "PC2", "PC3"],
    index = X_train.drop(["PC1", "PC2", "PC3", "O", "P"], axis = 1).columns
)

In [15]:
# Applying PCA to training set
X_val_scaled = (X_val.drop(["O", "P"], axis = 1) - X_val.drop(["O", "P"], axis = 1).mean(axis=0)) / X_val.drop(["O", "P"], axis = 1).std(axis=0)

X_pca = X_val_scaled.dot(loadings)

X_val = pd.concat((X_val, X_pca), axis = 1, join = "inner")


X_val["PC1"] = X_pca["PC1"]
X_val["PC2"] = X_pca["PC2"]
X_val["PC3"] = X_pca["PC3"]

In [18]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# Choosing Features
X1_train = X_train[["G", "M", "J", "C", "E", "H", "N", "PC1", "O", "P"]]
X1_val = X_val[["G", "M", "J", "C", "E", "H", "N", "PC1", "O", "P"]]

# Modelling
modelY1 = RandomForestRegressor(n_estimators=200, n_jobs = 4)

modelY1.fit(X1_train, y1_train)

# Predicting
y1_pred = modelY1.predict(X1_val)

# Choosing Features

X2_train = X_train[["A" ,"PC2", "K", "B", "D", "F", "I", "K", "L", "O", "P"]]
X2_val = X_val[["A" ,"PC2", "K", "B", "D", "F", "I", "K", "L", "O", "P"]]

# Modelling
modelY2 = RandomForestRegressor(n_estimators=200, n_jobs = 4)

modelY2.fit(X2_train, y2_train)

# Predicting
y2_pred = modelY2.predict(X2_val)


# OUTPUTS
print(f"Predicted score : {(r2_score(y2_pred, y2_val) + r2_score(y1_pred, y1_val))/2}")
print(f"Score Y1 : {r2_score(y1_pred, y1_val)} \nScore Y2 : {r2_score(y2_pred, y2_val)}")

Predicted score : 0.6371757096704411
Score Y1 : 0.672761075959635 
Score Y2 : 0.6015903433812473


In [20]:
from sklearn.decomposition import PCA

test_data = pd.read_csv('./data/test.csv') # Importing testing data
new_test_data = pd.read_csv('./data/test_new.csv') # Importing new testing data

test_data = pd.concat((test_data, new_test_data), axis = 1, join = "inner")

X = test_data.drop(["time", "id"], axis = 1) # Losing the time column

# Applying PCA to testing set
X_test = (X.drop(["O", "P"], axis = 1) - X.drop(["O", "P"], axis = 1).mean(axis=0)) / X.drop(["O", "P"], axis = 1).std(axis=0)

X_pca = X_test.dot(loadings)

X = pd.concat((X, X_pca), axis = 1, join = "inner")

X["PC1"] = X_pca["PC1"]
X["PC2"] = X_pca["PC1"]
X["PC3"] = X_pca["PC1"]

X1 = X[["G", "M", "J", "C", "E", "H", "N", "PC1", "O", "P"]]

y1_pred = pd.DataFrame(modelY1.predict(X1), index = test_data.id, columns = ["Y1"])

X2 = X[["A" ,"PC2", "K", "B", "D", "F", "I", "K", "L", "O", "P"]]

y2_pred = pd.DataFrame(modelY2.predict(X2), index = test_data.id, columns = ["Y2"])

out = pd.concat((y1_pred, y2_pred), axis = 1)

out.to_csv("./data/predictions.csv") # Scores 0.5888